![Week 4 Day 5](../assets/banners/w4d5.png)

# Week 4 Day 5 - 🧱 Project Day: The Transcription Subgraph, Tuned for Cost and Latency

It's project day! 🧱

This week you've collected every piece GradeWise needs to read a handwritten script: page images at the right resolution (Day 3), a prompt that never fixes the student, a mapper that turns "Ans 4(ii)" into `Q4b`, and confidence scores with a second reading for anything shaky (Day 4).

Today we snap those pieces together into one **LangGraph subgraph**: answer-sheet PDF in, `AnswerSheet` out. It reads every page **in parallel** with week 3's `Send`, verifies only what needs verifying, and logs what each sheet cost. Then we tune it, because Emma doesn't have one script, she has thirty, and next term she'll have two hundred.

![Week 4 roadmap](../assets/roadmap/week4.png)

### 🎯 By the end of this lab you will
1. 🧱 Read and draw the transcription subgraph: `render → transcribe_page ×N → map → verify_low_confidence`
2. 📝 Run it end to end on a real synthetic sheet, offline, and score it with CER against the ground truth
3. ⚡ Measure the speed-up of the parallel fan-out, and price a whole class at different settings
4. 🚀 Run this week's milestone script, `week4/project/transcription_graph.py`

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | 🧱 Day 5 - The Transcription Subgraph, Tuned for Cost and Latency | 🧪 |
| 2 | Day 5 - Week 4 Wrap-Up: Summary, Roadmap and Quiz | 🎬 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything up to Part 4 runs <b>offline</b> with the same fake vision model as yesterday: it replays canned transcriptions, but every other step (rendering, the parallel fan-out, mapping, verification, cost logging) is the real GradeWise code on the real PDFs. Part 4 then runs the milestone script with your real vision model: a fraction of a cent per sheet with the default <code>gpt-6-luna</code>, or $0 with <code>MODEL_MODE=free</code> (OpenRouter's free models: slower, and limited to 20 requests a minute and 50 a day, so use <code>DATA_SIZE=small</code>).</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT
from gradewise.console import success, warn

SYNTHETIC = DATA_DIR / "answer-sheets" / "synthetic"
ALL_SHEETS = {p.stem: p for p in sorted(SYNTHETIC.glob("student_*.pdf"))}  # every sheet on disk
# 📦 The class we loop over and send to a model: all 10 sheets, or student_01 + student_05 (Sam) with DATA_SIZE=small
SHEETS = {sid: ALL_SHEETS[sid] for sid in datasets.synthetic_students() if sid in ALL_SHEETS}
TRUTH_FILE = SYNTHETIC / "transcripts.json"
TRUTH = json.loads(TRUTH_FILE.read_text(encoding="utf-8")) if TRUTH_FILE.exists() else {}

if SHEETS and TRUTH:
    success(
        "Synthetic answer sheets found",
        {
            "Folder": str(SYNTHETIC.relative_to(REPO_ROOT)),
            "Sheets on disk": len(ALL_SHEETS),
            "Sheets in this run": len(SHEETS),
        },
    )
else:
    # ⚠️ Plan B: the sheets aren't there yet (run `uv run scripts/make_synthetic_sheets.py`, or `git pull`).
    #    We draw a small stand-in sheet so the rest of the lab still runs. Its text is our "ground truth".
    from PIL import Image, ImageDraw, ImageFont

    warn("data/answer-sheets/synthetic/ is missing or incomplete: using a small stand-in sheet instead.")
    standin = {
        "Q3": "Dedlock is when a process waits for a long time. Conditions: mutual exclution and circuler wait.",
        "Q4a": "When two threads use the same varible at the same time.",
    }
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/Inkfree.ttf", 34)
    except OSError:
        font = ImageFont.load_default()
    pages = []
    for label, qid in [("3)", "Q3"), ("4a)", "Q4a")]:
        img = Image.new("RGB", (1240, 1754), "white")
        draw = ImageDraw.Draw(img)
        words, lines, line = f"{label} {standin[qid]}".split(), [], ""
        for word in words:
            if len(line) + len(word) > 48:
                lines.append(line)
                line = ""
            line += word + " "
        for i, text in enumerate([*lines, line]):
            draw.text((90, 140 + i * 70), text, fill=(28, 45, 140), font=font)
        pages.append(img)
    target = OUTPUT_DIR / "week4" / "student_standin.pdf"
    target.parent.mkdir(parents=True, exist_ok=True)
    pages[0].save(target, "PDF", resolution=150, save_all=True, append_images=pages[1:])
    ALL_SHEETS = SHEETS = {"student_standin": target}
    TRUTH = {"student_standin": standin}

In [ ]:
from gradewise import datasets
from gradewise.schemas import ExamPaper

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
exam = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTION_IDS = [q.id for q in exam.questions]  # 💡 the mapper always needs every ID on the paper
# 📦 The questions we SCORE: all of them, or Q1–Q3 (all their parts) with DATA_SIZE=small
EVAL_IDS = [qid for qid in QUESTION_IDS if qid in set(datasets.exam_question_ids())]
print(f"{exam.title}: {len(QUESTION_IDS)} questions → {', '.join(QUESTION_IDS)}")
print(f"Scored in this run: {', '.join(EVAL_IDS)}")

In [ ]:
def show_table(rows: list[dict]) -> None:
    """Render a list of dicts as a Markdown table (a tiny stand-in for pandas: no extra dependencies)."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0])

    def cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ⏎ ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

In [ ]:
import re
import time

from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

from gradewise.transcription import AnswerSegment, PageTranscription


class FakeVisionLLM:
    """Stands in for a vision model: its structured output replays canned PageTranscriptions.

    💡 The same trick as tests/test_transcription.py. It reads the page number from the prompt,
       because parallel branches can call it in any order. `delay` fakes the network wait.
    """

    def __init__(self, first, second=None, delay: float = 0.0):
        self.first, self.second, self.delay = first, second or {}, delay
        self.calls: list[tuple[int, bool]] = []

    def with_structured_output(self, schema, **kwargs):
        return RunnableLambda(self._respond)

    def _respond(self, messages):
        time.sleep(self.delay)
        text = messages[-1].content[0]["text"]
        number = int(re.search(r"page (\d+)", text)[1])
        second = "second, independent reading" in text
        self.calls.append((number, second))
        pages = self.second if second and number in self.second else self.first
        parsed = pages.get(number, PageTranscription(page_number=number, confidence=0.9))
        raw = AIMessage(content="", usage_metadata={"input_tokens": 1500, "output_tokens": 250, "total_tokens": 1750})
        return {"raw": raw, "parsed": parsed, "parsing_error": None}

## Part 1: The subgraph

Open `gradewise/transcription/graph.py`. The whole pipeline is four nodes:

![Diagram](../assets/diagrams/56fc4035c364.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([START]) --> R[render<br/>PDF → page images]
  R -- "Send × N pages" --> T1[transcribe_page 1]
  R --> T2[transcribe_page 2]
  R --> T3[transcribe_page …]
  T1 & T2 & T3 --> M[map<br/>pages → one answer per question]
  M --> V[verify_low_confidence<br/>2nd pass at 300 DPI]
  V --> E([END])
```

</details>


| Node | Built on | What it does |
|---|---|---|
| `render` | Day 3 | `render_pages()` at `dpi` (200), then `preprocess()` each page |
| `transcribe_page` | Day 4 | One branch **per page**, launched with `Send`; each calls `transcribe_page_with_usage()` |
| `map` | Day 4 | `map_answers()`: waits for *all* branches, then builds the `AnswerSheet` |
| `verify_low_confidence` | Day 4 | `double_pass_verify()` on the pages behind low-confidence answers only |

🔁 **RECAP (week 3, Day 4):** two details make the fan-out safe. `fan_out()` returns one `Send("transcribe_page", {...})` per page, so the branches run in the same super-step, in parallel. And the state declares `pages: Annotated[list[PageTranscription], operator.add]`: a **reducer**, so each branch *adds* its page instead of overwriting the others.

In [ ]:
from gradewise.display import show_mermaid

from gradewise.transcription import build_transcription_graph
from gradewise.transcription.graph import TranscriptionState

print("State keys:", list(TranscriptionState.__annotations__))
app = build_transcription_graph(FakeVisionLLM({}))  # 💡 any model works for drawing: nothing is called
mermaid = app.get_graph().draw_mermaid()
show_mermaid(mermaid)

In [ ]:
display(Markdown(f"```mermaid\n{mermaid}\n```"))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">In the drawing, <code>render</code> connects to <code>transcribe_page</code> with a <b>dashed</b> edge, and there is only <i>one</i> <code>transcribe_page</code> node. Yet a 4-page script will run four transcriptions at once. Where does the "×4" come from, and why can't the diagram know it in advance?</span>
        </td>
    </tr>
</table>

## Part 2: One sheet, end to end

`transcribe_sheet(pdf_path, exam, student_id, llm=None, dpi=200, verify=True)` builds the graph, runs it and prints a cost summary. Let's give it a **perfect** fake model: one that reads exactly the ground truth, spread over the sheet's real pages. If the plumbing is right, every answer must come out with a CER of 0.

In [ ]:
from gradewise.transcription import cer, transcribe_sheet
from gradewise.transcription.pdf_to_images import page_count

SHEET = "student_07" if "student_07" in ALL_SHEETS else next(iter(ALL_SHEETS))  # 💡 offline, so any sheet on disk
truth = TRUTH[SHEET]
n_pages = page_count(ALL_SHEETS[SHEET])


def canned_pages(answers: dict[str, str], conf: dict[str, float] | None = None) -> dict[int, PageTranscription]:
    """Spread the answers over the sheet's pages, in order, like a model reading the real sheet."""
    items = list(answers.items())
    per_page = -(-len(items) // n_pages)  # ceiling division
    return {
        n: PageTranscription(
            page_number=n,
            confidence=0.9,
            segments=[
                AnswerSegment(label=q, text=t, confidence=(conf or {}).get(q, 0.93))
                for q, t in items[(n - 1) * per_page : n * per_page]
            ],
        )
        for n in range(1, n_pages + 1)
    }


perfect = FakeVisionLLM(canned_pages(truth))
sheet = transcribe_sheet(ALL_SHEETS[SHEET], exam, SHEET, llm=perfect, dpi=150)

In [ ]:
show_table(
    [
        {
            "question": a.question_id,
            "pages": a.pages,
            "confidence": f"{a.confidence:.2f}",
            "CER vs truth": f"{cer(truth[a.question_id], a.text):.1%}" if a.question_id in truth else "–",
            "text": a.text[:55] or "(blank)",
        }
        for a in sheet.answers
    ]
)

Every answer the student wrote has a CER of 0.0%, and the questions they skipped are reported as **blank**, never filled in.

A perfect model is a good test of the plumbing but a poor test of the *checks*. Real models stumble. Let's give our fake a realistic bad day: it can't read part of one answer (`[illegible]`, low confidence), and the independent second reading gets it right.

In [ ]:
shaky_q = next(iter(truth))
words = truth[shaky_q].split()
shaky_text = " ".join(["[illegible]" if i % 4 == 1 else w for i, w in enumerate(words)])

first_pass = canned_pages({**truth, shaky_q: shaky_text}, conf={shaky_q: 0.62})
second_pass = canned_pages(truth, conf={shaky_q: 0.90})
fake = FakeVisionLLM(first_pass, second_pass)

state = build_transcription_graph(fake).invoke(
    {"pdf_path": str(ALL_SHEETS[SHEET]), "exam": exam, "student_id": SHEET, "dpi": 150}
)
print("vision calls (page, second pass?):", sorted(fake.calls))
for check in state["checks"]:
    print(
        f"\n🔁 {check.question_id}: first {check.first_confidence:.2f} → second {check.second_confidence:.2f} · "
        f"CER between passes {check.cer_between_passes:.0%} · final {check.final_confidence:.2f} · "
        f"kept pass {check.kept_pass} · flagged {check.flagged}"
    )
fixed = state["sheet"].answer_for(shaky_q)
print(f"   final text CER vs truth: {cer(truth[shaky_q], fixed.text):.1%}")

Read the call log: every page once in the first pass, then **one** page again for the second pass, the one behind the shaky answer. The two readings disagree a lot (all those `[illegible]` markers), so even though the second pass was confident and correct, the answer stays **flagged**: GradeWise keeps the better text, but a human confirms it. That's the right call when marks are at stake.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">This is the "human in the loop" in its most cost-effective form. Instead of Emma re-reading 30 scripts, she checks the handful of answers GradeWise is unsure about, with both readings and the page image side by side. Confidence-based triage like this is how document-AI products in insurance, banking and healthcare keep accuracy high without paying humans to re-check everything.</span>
        </td>
    </tr>
</table>

## Part 3: Tuning for cost and latency

### ⚡ Latency: the parallel fan-out

Real vision calls take seconds. Our fake can pretend: `delay=0.5` makes every call wait half a second. We transcribe a 4-page sheet twice, once with one page at a time and once with GradeWise's default of up to `MAX_PARALLEL_PAGES` (4) at once:

In [ ]:
from gradewise.transcription.graph import MAX_PARALLEL_PAGES

LONG = max(SHEETS, key=lambda s: page_count(SHEETS[s]))
slow_fake = FakeVisionLLM(canned_pages(TRUTH[LONG]), delay=0.5)
app = build_transcription_graph(slow_fake, verify=False)
inputs = {"pdf_path": str(SHEETS[LONG]), "exam": exam, "student_id": LONG, "dpi": 100}

timings = []
for concurrency in (1, MAX_PARALLEL_PAGES):
    start = time.perf_counter()
    app.invoke(inputs, config={"max_concurrency": concurrency})
    timings.append(
        {
            "max_concurrency": concurrency,
            "pages": page_count(SHEETS[LONG]),
            "wall time": f"{time.perf_counter() - start:.2f} s",
        }
    )
show_table(timings)

Same work, a fraction of the wait. 🚀 **PRO:** the cap matters as much as the parallelism. Thirty scripts × four pages fired at once is a quick way to hit your provider's rate limit; `max_concurrency` keeps GradeWise polite. With `MODEL_MODE=free` the package goes further: it lowers parallelism to 2 and retries 429 ("too many requests") errors with backoff, because OpenRouter's free models allow only 20 requests a minute (and 50 a day on an account that has never bought $10 of credits).

### 💰 Cost: the knobs

| Knob | Where | Effect |
|---|---|---|
| `dpi` | `transcribe_sheet(dpi=...)` | Render sharpness. Above ~200 mostly costs render time (Day 3) |
| `MAX_SIDE` | `pdf_to_images.py` | The pixels you actually send: the biggest lever on image tokens |
| `verify` / `threshold` | `transcribe_sheet(verify=..., threshold=...)` | How many pages get a second, pricier reading |
| `verify_dpi` | `build_transcription_graph(verify_dpi=...)` | Sharpness of the second reading (300 DPI, 2400 px cap) |
| `MODEL_MODE` (or `VISION_MODEL`, with `MODEL_MODE=custom`) | `.env` | The price per token: `cheap` = `gpt-6-luna` ($0.10 in / $0.50 out per 1M tokens), `free` = `openrouter/free`, OpenRouter's router over free models ($0, rate-limited). The week 4 exercise compares two models |

Let's price the whole class. We render every real page at 200 DPI, count image tokens with both of Day 3's rules (OpenAI-style tiles via `estimate_image_tokens()`, and the rough area rule), add ~900 tokens of prompt and ~300 of output per page, and price it with `gradewise/pricing.py` at your vision model's list price (`gpt-6-luna` with the default `MODEL_MODE=cheap`: $0.10 in / $0.50 out per million tokens; `MODEL_MODE=free` prints $0, but the token columns still tell the story). 📦 "The class" is whatever `DATA_SIZE` says: all 10 sheets, or 2 with `small` (the per-sheet column is what to compare):

In [ ]:
from gradewise.pricing import PRICES_VERIFIED, chat_cost
from gradewise.transcription import preprocess, render_pages
from gradewise.transcription.pdf_to_images import estimate_image_tokens

PROMPT_TOKENS, OUTPUT_TOKENS, REREAD_SHARE = 900, 300, 0.2  # assume 20% of pages get a second reading
print(f"💰 Priced for {settings.transcription_model} (list prices checked {PRICES_VERIFIED})")


def price(tokens_in: int, tokens_out: int) -> float:
    return chat_cost(tokens_in, tokens_out, model=settings.transcription_model)


def area_tokens(img) -> int:
    """Day 3's rough 'pixels / 750' rule (no provider-side downscale here, to show the raw trend)."""
    return round(img.width * img.height / 750)


all_pages = [img for path in SHEETS.values() for img in render_pages(path, dpi=200)]
rows = []
for max_side in (800, 1600, 2400):
    capped = [preprocess(img, max_side=max_side) for img in all_pages]
    for rule, count in (("tiles", estimate_image_tokens), ("area", area_tokens)):
        tokens_in = sum(count(img) for img in capped) + PROMPT_TOKENS * len(capped)
        first = price(tokens_in, OUTPUT_TOKENS * len(capped))
        rows.append(
            {
                "size cap": f"{max_side} px",
                "token rule": rule,
                "pages": len(capped),
                "input tokens": f"{tokens_in:,}",
                "first pass": f"${first:.4f}",
                "with verification": f"${first * (1 + REREAD_SHARE):.4f}",
                "per sheet": f"${first * (1 + REREAD_SHARE) / len(SHEETS):.4f}",
            }
        )
show_table(rows)

Under the **tiles** rule, the 1600 px and 2400 px caps cost exactly the same: the provider shrinks every page to 768 px on its short side anyway, so the extra pixels only cost upload time. Under the **area** rule, a bigger cap means proportionally more tokens. Either way the lesson from Day 3 holds at class scale: the **size cap** is the lever, not the DPI, and the second-pass verification adds only ~20% because it re-reads just the shaky pages.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">These are list prices from <code>gradewise/pricing.py</code> (checked 2026-11-12; the same table prices the cost line <code>transcribe_sheet()</code> prints), and the token rule is one provider's: check your provider's pricing page before you quote a number to anyone. With <code>MODEL_MODE=free</code> the bill is $0, but you get 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), a different free model on each call (so CER varies run to run), and free providers may log or train on prompts: fine for the fictional sheets, never for real students' scripts. Notice too that <b>output tokens</b> cost five times as much as input here: a verbose model that adds commentary to every page costs real money. Structured output keeps replies lean.</span>
        </td>
    </tr>
</table>

## Part 4: 🚀 Run the milestone

This week's milestone script is `week4/project/transcription_graph.py`. Open it: it loads the exam, calls `transcribe_sheet()`, prints a table (with a **CER** column whenever `transcripts.json` sits next to the PDF) and saves the `AnswerSheet` to `output/transcriptions/<student>.json` for week 5.

First, a free dress rehearsal. We load the script as a module and swap `gradewise.models.vision_model` for our fake for the duration of one call:

In [ ]:
import importlib.util

import gradewise.models

script = REPO_ROOT / "week4" / "project" / "transcription_graph.py"
spec = importlib.util.spec_from_file_location("week4_transcription_demo", script)
demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(demo)

real_factory = gradewise.models.vision_model
gradewise.models.vision_model = lambda **kwargs: FakeVisionLLM(first_pass, second_pass)  # 🎭 rehearsal only
try:
    demo.main(pdf=ALL_SHEETS[SHEET], exam_path=EXAM_FILE, dpi=150, no_verify=False)
finally:
    gradewise.models.vision_model = real_factory  # ⚠️ always put the real factory back

In [ ]:
saved = OUTPUT_DIR / "transcriptions" / f"{SHEET}.json"
print(saved.read_text(encoding="utf-8")[:700], "…")

Now the real thing, exactly as you'd run it in a terminal:

```bash
uv run week4/project/transcription_graph.py data/answer-sheets/synthetic/student_07.pdf
```

The next cell runs the same command from Python (so it works the same on Windows, Mac and Linux), on `student_07`, or on `student_01` with `DATA_SIZE=small`. With a key, you'll see the real cost panel and the real CER of your vision model; without one, a friendly *Missing API key* panel.

In [ ]:
import subprocess
import sys

REAL_SHEET = SHEET if SHEET in SHEETS else next(iter(SHEETS))  # 📦 real calls stay inside this run's class
result = subprocess.run(
    [sys.executable, str(script), str(SHEETS[REAL_SHEET]), "--dpi", "200"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"},  # 💡 one adjustment for Windows: emoji-safe output
)
print(result.stdout[-4000:] or result.stderr[-2000:])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">🎉 <b>Milestone M4 reached.</b> GradeWise can now read a handwritten answer sheet: <code>gradewise/transcription/graph.py</code> gives week 5 an <code>AnswerSheet</code> with verbatim text, pages and a confidence for every question, plus a list of what needs a human. Together with Day 2's <code>find_evidence()</code>, the grader has everything it needs: <b>what the student wrote</b>, and <b>what the notes say</b>.</span>
        </td>
    </tr>
</table>

## Part 5: Week 4 wrap-up

### What GradeWise learned this week

| Day | You built | Lives in |
|:-:|---|---|
| 1 | A corrective agentic RAG loop: route → retrieve → grade → rewrite → answer | `gradewise/rag/agentic_rag.py` |
| 2 | Grounded answers + the evidence retriever, built from the examiner's words | `gradewise/rag/evidence.py` |
| 3 | PDF pages → clean, right-sized images; the DPI/token trade-off | `gradewise/transcription/pdf_to_images.py` |
| 4 | Verbatim transcription, label mapping, confidence and double-pass checks | `vision.py`, `mapper.py`, `confidence.py` |
| 5 | 🧱 The transcription subgraph, in parallel, with cost logging | `gradewise/transcription/graph.py` |

### 🗺️ Where we are heading

![Diagram](../assets/diagrams/27c7a2ba17b4.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  W1[W1 Exam → JSON] --> W2[W2 Knowledge base] --> W3[W3 Agents & graphs] --> W4[W4 Evidence + handwriting]:::here --> W5[W5 Grading & reports] --> W6[W6 Ship it]
  classDef here fill:#7c5cff,color:#fff
```

</details>


**Next week** is the one the whole course has been building towards: the **grading agent**. For every question it gets Sam's verbatim answer (today), the marking criteria (week 1) and the evidence from the notes (Day 2), and it awards marks criterion by criterion, with a citation for every decision. Then come report cards, and an honest evaluation against human examiners' marks.

📝 **Quiz 4** is waiting for you on Udemy (10 questions): it's a great way to check that this week's ideas have landed.

## Recap, and where we are heading

Today you:
- 🧱 read and drew the transcription subgraph, and saw how `Send` + a reducer make the page fan-out safe
- 📝 ran it on a real synthetic sheet, scored it with CER, and watched double-pass verification catch a shaky answer
- ⚡ measured the parallel speed-up and priced a whole class at three size caps
- 🚀 ran the week 4 milestone script, first as a free rehearsal, then for real

⏭ **Next:** the week 4 exercise below, then `week5/day1.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">This week's big exercise is in <code>week4/week4_EXERCISE.ipynb</code>: <b>compare two vision models on CER and cost</b>. You'll transcribe the same synthetic sheets with two models (your default vision model, <code>gpt-6-luna</code>, against the stronger and pricier <code>gpt-6.1-terra</code>, or OpenRouter's free router, <code>openrouter/free</code>), score each against the ground truth, log tokens, cost and wall time, and decide which one Emma should use. It runs in a free dry-run mode first, so you can build it before spending anything. Solution: <code>week4/solutions/week4_EXERCISE_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ec4899;">Thank you</h2>
            <span style="color:#ec4899;">That's week 4 done, and it was a big one: GradeWise can now find evidence <i>and</i> read handwriting. Thank you for sticking with it! If something didn't work, the Udemy Q&A is the place to ask, and if you've built something cool on top of this week, share it in <code>community_contributions/</code>. See you in week 5, where it all comes together. 🎓</span>
        </td>
    </tr>
</table>